# 🌷 LILY H3 VIDEO STUDIO — KAGGLE

**H3 only.** No LTX, Wan 2.2, or Hunyuan downloads. Enable **T4 x2** before Cell 1, then run **1 → 2 → 3 → 4**. Cell 3 is the long H3 download/verification. Every code cell recreates its own paths, so the old `ROOT is not defined` failure cannot happen.


## 1 — 🏗️ Setup WanGP + storage


In [ ]:
from pathlib import Path
import os, subprocess, shutil
ROOT=Path('/kaggle/working/Wan2GP'); DATA=Path('/kaggle/temp/Wan2GP-data'); CKPTS=DATA/'ckpts'; LORAS=DATA/'loras'; CACHE=DATA/'cache'; OUTPUTS=Path('/kaggle/working/Wan2GP-outputs')
if shutil.which('nvidia-smi') is None: raise RuntimeError('No NVIDIA GPU. Kaggle → Settings → Accelerator → GPU / T4 x2, let it restart, then rerun Cell 1.')
subprocess.run(['nvidia-smi'],check=True)
for p in (CKPTS,LORAS,CACHE,OUTPUTS): p.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(CACHE/'huggingface'); os.environ['HUGGINGFACE_HUB_CACHE']=str(CACHE/'huggingface'/'hub'); os.environ['TRANSFORMERS_CACHE']=str(CACHE/'huggingface'/'transformers'); os.environ['TORCH_HOME']=str(CACHE/'torch'); os.environ['XDG_CACHE_HOME']=str(CACHE/'.cache'); os.environ['WAN_CACHE_DIR']=str(CACHE); os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
def free(): return shutil.disk_usage('/kaggle').free/1024**3
print(f'💾 Free disk before setup: {free():.1f} GiB')
repo='https://github.com/deepbeepmeep/Wan2GP.git'
if not (ROOT/'.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1',repo,str(ROOT)],check=True)
else: subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
def attach(src,dst):
    dst.mkdir(parents=True,exist_ok=True)
    if src.is_symlink():
        try:
            if src.resolve()==dst.resolve(): return
        except FileNotFoundError: pass
        src.unlink()
    elif src.exists():
        for x in list(src.iterdir()):
            y=dst/x.name
            if not y.exists(): shutil.move(str(x),str(y))
            elif x.is_dir(): shutil.rmtree(x)
            else: x.unlink()
        shutil.rmtree(src)
    src.symlink_to(dst,target_is_directory=True)
attach(ROOT/'ckpts',CKPTS); attach(ROOT/'loras',LORAS); attach(ROOT/'outputs',OUTPUTS)
env=os.environ.copy(); env['DEBIAN_FRONTEND']='noninteractive'; prefix=[] if os.geteuid()==0 else ['sudo']
subprocess.run(prefix+['apt-get','update','-qq'],check=True,env=env)
subprocess.run(prefix+['apt-get','install','-y','--no-install-recommends','ffmpeg','libglib2.0-0','libgl1','libportaudio2'],check=True,env=env)
print(f'💾 Free disk after setup: {free():.1f} GiB')
print('✅ CELL 1 COMPLETE')


## 2 — 🧪 Install + validate dependencies


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
ROOT=Path('/kaggle/working/Wan2GP'); CACHE=Path('/kaggle/temp/Wan2GP-data/cache')
if not (ROOT/'.git').exists(): raise RuntimeError('WanGP is missing. Run Cell 1 first; if Kaggle reset, start again from Cell 1.')
CACHE.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(CACHE/'huggingface'); os.environ['HUGGINGFACE_HUB_CACHE']=str(CACHE/'huggingface'/'hub'); os.environ['TRANSFORMERS_CACHE']=str(CACHE/'huggingface'/'transformers'); os.environ['TORCH_HOME']=str(CACHE/'torch'); os.environ['XDG_CACHE_HOME']=str(CACHE/'.cache'); os.environ['WAN_CACHE_DIR']=str(CACHE); os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
probe="""import json\nout={}\nfor n in ('torch','torchvision','torchaudio'):\n try:\n  m=__import__(n); out[n]=getattr(m,'__version__',None)\n except Exception: out[n]=None\nprint(json.dumps(out))"""
raw=subprocess.check_output([sys.executable,'-c',probe],text=True); installed=json.loads(raw.strip().splitlines()[-1])
if not installed.get('torch'): raise RuntimeError('Kaggle CUDA Torch not detected. Confirm T4 x2 and restart from Cell 1.')
constraints=CACHE/'kaggle-torch-constraints.txt'; pins=[f"torch=={installed['torch'].split('+',1)[0]}"]
for n in ('torchvision','torchaudio'):
    if installed.get(n): pins.append(f"{n}=={installed[n].split('+',1)[0]}")
constraints.write_text('\n'.join(pins)+'\n'); print('🔒 Protecting CUDA packages:\n'+constraints.read_text())
env=os.environ.copy(); env['PIP_NO_CACHE_DIR']='1'; env['PIP_DISABLE_PIP_VERSION_CHECK']='1'
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade','setuptools','wheel'],check=True,env=env)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade-strategy','only-if-needed','-r',str(ROOT/'requirements.txt'),'-c',str(constraints)],check=True,env=env)
t=ROOT/'preprocessing/matanyone/tools/interact_tools.py'
if t.exists():
    old=t.read_text(); new=old.replace("matplotlib.use('TkAgg')","matplotlib.use('Agg')")
    if new!=old: t.write_text(new)
validate="""import numpy,torch,mmgp,rembg,gradio\nassert torch.cuda.is_available()\nprint('NumPy:',numpy.__version__)\nprint('Torch:',torch.__version__)\nprint('CUDA:',torch.version.cuda)\nprint('GPU:',torch.cuda.get_device_name(0))\nprint('GPU count:',torch.cuda.device_count())\nprint('mmgp/rembg/gradio: OK')"""
subprocess.run([sys.executable,'-c',validate],check=True,env=os.environ.copy())
print(f"💾 Free disk after dependencies: {shutil.disk_usage('/kaggle').free/1024**3:.1f} GiB")
print('✅ CELL 2 COMPLETE')


## 3 — 📦 Download + verify MiniMax H3 FL2VA Pruned

This is the only model download. It checks disk space before starting and dynamically discovers the current H3 FL2VA Pruned model exposed by WanGP.


In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request
ROOT=Path('/kaggle/working/Wan2GP'); HELPER=Path('/kaggle/working/lily_h3_prewarm.py')
if not (ROOT/'.git').exists(): raise RuntimeError('WanGP is missing. Run Cells 1 and 2 first; if Kaggle reset, start again from Cell 1.')
url='https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_h3_prewarm.py'
urllib.request.urlretrieve(url,HELPER); print('📦 H3 prewarm helper updated from GitHub.')
subprocess.run([sys.executable,'-u',str(HELPER)],cwd=str(ROOT),env=os.environ.copy(),check=True)
print('✅ CELL 3 COMPLETE')


## 4 — 🌷 Launch Lily H3 Video Studio

H3 only. If Cell 3 did not finish, the UI refuses to silently redownload the model.


In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request
ROOT=Path('/kaggle/working/Wan2GP'); STUDIO=Path('/kaggle/working/lily_h3_studio.py')
if not (ROOT/'.git').exists(): raise RuntimeError('WanGP is missing. Kaggle reset the runtime; start again from Cell 1.')
url='https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_h3_studio.py'
urllib.request.urlretrieve(url,STUDIO); print('🌷 H3 Studio updated from GitHub.')
print('Launching… keep this cell running and tap the public gradio.live link when it appears.')
subprocess.run([sys.executable,'-u',str(STUDIO)],cwd=str(ROOT),env=os.environ.copy(),check=False)
